# 15 — Ecuación OLS, matriz de diseño e interpretación de coeficientes

Esta notebook no evalúa performance predictiva. Para eso están las notebooks de comparación de modelos, errores por decil, estructura temporal/geográfica y distribución.

La pregunta acá es más básica y más econométrica:

> **¿Qué ecuación OLS estamos estimando exactamente, sobre qué matriz de diseño, con qué unidades de interpretación y contra qué categorías de referencia?**

La notebook usa los diagnostics nuevos cuando están disponibles:

- `coefficients_transformed_all.csv`
- `design_matrix_features.csv`
- `onehot_reference_levels.csv`
- `fixed_effect_reference_levels.csv`
- `coefficient_family_summary.csv`
- `top_coefficients_by_family.csv`
- `feature_metadata.json`

La regla editorial de esta notebook es deliberada: **casi no hay figuras**. Un gráfico sólo entra si aclara una pregunta metodológica. La mayoría de los resultados importantes son tablas interpretables.


## 00. Setup y política de especificaciones

Usamos tres especificaciones con roles distintos:

1. **`ols_core`**: benchmark limpio. Incluye demografía, educación, trabajo y hogar/vivienda. No incluye pirámide del hogar ni fixed effects.
2. **`ols_core_aglo_plus_time_fe`**: especificación ajustada. Misma base limpia, más FE de aglomerado, año y trimestre.
3. **`ols_core_with_pyramid`**: sensibilidad composicional. Sirve para ver qué cambia al agregar pirámide del hogar, pero no es el default interpretativo.

Si existen runs legacy, la notebook los puede leer, pero el análisis central se basa en `ols_core` y `ols_core_aglo_plus_time_fe`.


In [1]:
from pathlib import Path
import json
import yaml
import re
from collections.abc import Mapping

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 160)
pd.set_option("display.width", 180)

ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/home/matias/repos/income-modeling-eph"),
]

ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "reports" / "runs").exists()),
    Path("/home/matias/repos/income-modeling-eph"),
)

RUNS_DIR = ROOT / "reports" / "runs"
CONFIGS_DIR = ROOT / "configs"
FEATURE_CONTRACT_PATH = CONFIGS_DIR / "feature_contract.yaml"

OUTPUT_DIR = ROOT / "reports" / "notebook_outputs" / "ols_equation_and_coefficients_v2"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Figures are disabled by default in this notebook. Turn on only if the coefficient
# stability scatter genuinely helps after inspecting the tables.
MAKE_OPTIONAL_FIGURES = False

MAIN_EXPERIMENTS = [
    "ols_core",
    "ols_core_aglo_plus_time_fe",
]

SENSITIVITY_EXPERIMENTS = [
    "ols_core_with_pyramid",
]

LEGACY_OR_AUX_EXPERIMENTS = [
    "ols_core_no_pyramid",
    "ols_core_aglo_fe",
    "ols_core_year_fe",
    "ols_core_quarter_fe",
    "ols_core_year_plus_quarter_fe",
]

EXPERIMENT_ORDER = MAIN_EXPERIMENTS + SENSITIVITY_EXPERIMENTS + LEGACY_OR_AUX_EXPERIMENTS

EXPERIMENT_LABELS = {
    "ols_core": "Core OLS limpio",
    "ols_core_aglo_plus_time_fe": "Core + aglo + año + trimestre FE",
    "ols_core_with_pyramid": "Core + pirámide del hogar",
    "ols_core_no_pyramid": "Core sin pirámide (legacy)",
    "ols_core_aglo_fe": "Core + aglomerado FE (legacy/aux)",
    "ols_core_year_fe": "Core + año FE (aux)",
    "ols_core_quarter_fe": "Core + trimestre FE (aux)",
    "ols_core_year_plus_quarter_fe": "Core + año + trimestre FE (aux)",
}

AGLO_LABELS = {
    "0": "No aplica",
    "2": "La Plata",
    "3": "Bahía Blanca",
    "4": "Rosario",
    "5": "Santa Fe",
    "6": "Paraná",
    "7": "Posadas",
    "8": "Resistencia",
    "9": "Comodoro/Rada Tilly",
    "10": "Mendoza",
    "12": "Corrientes",
    "13": "Córdoba",
    "14": "Concordia",
    "15": "Formosa",
    "17": "Neuquén",
    "18": "Sgo.-La Banda",
    "19": "Jujuy-Palpalá",
    "20": "Río Gallegos",
    "22": "Catamarca",
    "23": "Salta",
    "25": "La Rioja",
    "26": "San Luis",
    "27": "San Juan",
    "29": "Tucumán",
    "30": "Santa Rosa-Toay",
    "31": "Ushuaia-Río Grande",
    "32": "CABA",
    "33": "GBA",
    "34": "Mar del Plata",
    "36": "Río Cuarto",
    "38": "San Nicolás-VC",
    "91": "Rawson-Trelew",
    "93": "Viedma-C. Patagones",
    "99": "Missing",
}

print("ROOT:", ROOT)
print("RUNS_DIR:", RUNS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


ROOT: /home/matias/repos/income-modeling-eph
RUNS_DIR: /home/matias/repos/income-modeling-eph/reports/runs
OUTPUT_DIR: /home/matias/repos/income-modeling-eph/reports/notebook_outputs/ols_equation_and_coefficients_v2


## 01. Utilidades y descubrimiento de runs

La selección de runs se hace por `config_used.yaml` y `experiment.id`. Esto evita el bug clásico de confundir `ols_core_*` con cualquier experimento cuyo nombre empieza por `ols_core`.


In [2]:
def read_json_if_exists(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def read_yaml_if_exists(path: Path):
    if not path.exists():
        return None
    return yaml.safe_load(path.read_text(encoding="utf-8"))


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def as_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


def safe_json_load(value):
    if pd.isna(value):
        return None
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("[") or text.startswith("{"):
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                return value
    return value


def discover_latest_run(experiment_id: str) -> Path | None:
    if not RUNS_DIR.exists():
        return None

    exact = []
    loose = []
    for candidate in sorted(RUNS_DIR.glob(f"{experiment_id}_*"), key=lambda p: p.name):
        if not candidate.is_dir():
            continue
        loose.append(candidate)
        config = read_yaml_if_exists(candidate / "config_used.yaml") or {}
        current_id = ((config.get("experiment") or {}).get("id"))
        if current_id == experiment_id:
            exact.append(candidate)

    if exact:
        return exact[-1]
    if loose:
        return loose[-1]
    return None


def load_run_info(experiment_id: str) -> dict:
    run_dir = discover_latest_run(experiment_id)
    if run_dir is None:
        return {
            "experiment": experiment_id,
            "label": EXPERIMENT_LABELS.get(experiment_id, experiment_id),
            "found": False,
            "run_dir": None,
        }

    config = read_yaml_if_exists(run_dir / "config_used.yaml") or {}
    manifest = read_json_if_exists(run_dir / "run_manifest.json") or {}
    dataset_card = read_json_if_exists(run_dir / "dataset_card.json") or {}
    feature_columns = read_json_if_exists(run_dir / "feature_columns.json") or []
    feature_metadata = read_json_if_exists(run_dir / "artifacts" / "feature_metadata.json") or {}
    comparison = read_csv_if_exists(run_dir / "metrics" / "model_comparison.csv")

    feature_view = config.get("feature_view") or {}
    model_design = config.get("model_design") or {}
    fixed_effects = model_design.get("fixed_effects") or []

    return {
        "experiment": experiment_id,
        "label": EXPERIMENT_LABELS.get(experiment_id, experiment_id),
        "found": True,
        "run_dir": run_dir,
        "run_id": manifest.get("run_id") or run_dir.name,
        "config": config,
        "manifest": manifest,
        "dataset_card": dataset_card,
        "feature_columns": feature_columns,
        "feature_metadata": feature_metadata,
        "model_comparison": comparison,
        "include_blocks": feature_view.get("include_blocks"),
        "drop_columns": feature_view.get("drop_columns"),
        "fixed_effects": fixed_effects,
        "diagnostics_dir": run_dir / "diagnostics",
        "artifacts_dir": run_dir / "artifacts",
    }


RUN_INFO = {exp: load_run_info(exp) for exp in EXPERIMENT_ORDER}
run_selection = pd.DataFrame([
    {
        "experiment": exp,
        "label": info["label"],
        "found": info["found"],
        "run_id": info.get("run_id"),
        "run_dir": str(info.get("run_dir")) if info.get("run_dir") else None,
        "include_blocks": info.get("include_blocks"),
        "fixed_effects": info.get("fixed_effects"),
    }
    for exp, info in RUN_INFO.items()
])

run_selection.to_csv(TABLE_DIR / "T1_run_selection.csv", index=False)

found_main = [exp for exp in MAIN_EXPERIMENTS if RUN_INFO[exp]["found"]]
if not found_main:
    raise RuntimeError(
        "No main OLS runs found. Run at least ols_core and preferably ols_core_aglo_plus_time_fe."
    )

run_selection


,experiment,label,found,run_id,run_dir,include_blocks,fixed_effects
0,ols_core,Core OLS limpio,True,ols_core_20260611T084128Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...",[]
1,ols_core_aglo_plus_time_fe,Core + aglo + año + trimestre FE,True,ols_core_aglo_plus_time_fe_20260611T085155Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...","[{'name': 'aglo_fe', 'columns': ['AGLOMERADO']..."
2,ols_core_with_pyramid,Core + pirámide del hogar,True,ols_core_with_pyramid_20260611T084159Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...",[]
3,ols_core_no_pyramid,Core sin pirámide (legacy),True,ols_core_no_pyramid_20260611T050802Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...",[]
4,ols_core_aglo_fe,Core + aglomerado FE (legacy/aux),True,ols_core_aglo_fe_20260611T085123Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...","[{'name': 'aglo_fe', 'columns': ['AGLOMERADO']..."
5,ols_core_year_fe,Core + año FE (aux),True,ols_core_year_fe_20260611T084828Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...","[{'name': 'year_fe', 'columns': ['ANO4'], 'max..."
6,ols_core_quarter_fe,Core + trimestre FE (aux),True,ols_core_quarter_fe_20260611T084901Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...","[{'name': 'quarter_fe', 'columns': ['TRIMESTRE..."
7,ols_core_year_plus_quarter_fe,Core + año + trimestre FE (aux),True,ols_core_year_plus_quarter_fe_20260611T084930Z,/home/matias/repos/income-modeling-eph/reports...,"[demographic, education, labor, housing_househ...","[{'name': 'year_fe', 'columns': ['ANO4'], 'max..."


## 02. Política metodológica: qué cuenta como benchmark OLS

La especificación central no debe absorber todo lo que “mejora R²”. Un benchmark interpretativo necesita una frontera clara:

- **Core limpio**: covariables individuales y del hogar observables.
- **Ajustado con FE**: mismo core más heterogeneidad aditiva por lugar y tiempo.
- **Pirámide**: sensibilidad composicional, no default.

Esto evita mezclar tres cosas distintas: capacidad predictiva, control de heterogeneidad de grupo e ingeniería de features.


In [3]:
def has_pyramid_blocks(include_blocks) -> bool:
    blocks = [str(b) for b in as_list(include_blocks)]
    return any("pyramid" in b.lower() for b in blocks)


def fe_summary_text(fixed_effects) -> str:
    specs = as_list(fixed_effects)
    pieces = []
    for spec in specs:
        if not isinstance(spec, Mapping):
            continue
        name = spec.get("name")
        cols = ", ".join(str(c) for c in as_list(spec.get("columns")))
        pieces.append(f"{name}({cols})")
    return "; ".join(pieces) if pieces else "none"


spec_policy_rows = []
for exp, info in RUN_INFO.items():
    if not info["found"]:
        continue
    role = (
        "benchmark_clean"
        if exp == "ols_core"
        else "adjusted_benchmark"
        if exp == "ols_core_aglo_plus_time_fe"
        else "sensitivity_or_auxiliary"
    )
    include_blocks = info.get("include_blocks")
    fixed_effects = info.get("fixed_effects")
    spec_policy_rows.append({
        "experiment": exp,
        "label": info["label"],
        "role": role,
        "uses_household_pyramid": has_pyramid_blocks(include_blocks),
        "include_blocks": include_blocks,
        "fixed_effects": fe_summary_text(fixed_effects),
    })

spec_policy = pd.DataFrame(spec_policy_rows)
spec_policy.to_csv(TABLE_DIR / "T2_specification_policy.csv", index=False)
spec_policy


,experiment,label,role,uses_household_pyramid,include_blocks,fixed_effects
0,ols_core,Core OLS limpio,benchmark_clean,False,"[demographic, education, labor, housing_househ...",none
1,ols_core_aglo_plus_time_fe,Core + aglo + año + trimestre FE,adjusted_benchmark,False,"[demographic, education, labor, housing_househ...",aglo_fe(AGLOMERADO); year_fe(ANO4); quarter_fe...
2,ols_core_with_pyramid,Core + pirámide del hogar,sensitivity_or_auxiliary,True,"[demographic, education, labor, housing_househ...",none
3,ols_core_no_pyramid,Core sin pirámide (legacy),sensitivity_or_auxiliary,False,"[demographic, education, labor, housing_househ...",none
4,ols_core_aglo_fe,Core + aglomerado FE (legacy/aux),sensitivity_or_auxiliary,False,"[demographic, education, labor, housing_househ...",aglo_fe(AGLOMERADO)
5,ols_core_year_fe,Core + año FE (aux),sensitivity_or_auxiliary,False,"[demographic, education, labor, housing_househ...",year_fe(ANO4)
6,ols_core_quarter_fe,Core + trimestre FE (aux),sensitivity_or_auxiliary,False,"[demographic, education, labor, housing_househ...",quarter_fe(TRIMESTRE)
7,ols_core_year_plus_quarter_fe,Core + año + trimestre FE (aux),sensitivity_or_auxiliary,False,"[demographic, education, labor, housing_househ...",year_fe(ANO4); quarter_fe(TRIMESTRE)


## 03. Ecuación estimada

La forma compacta `y_i = α + X_iβ + ε_i` es demasiado pobre para esta tesis. La ecuación real depende de la matriz transformada:

$$
\log_{10}(P47T_i)
=
\alpha
+ X^{num}_i \beta
+ B_i \gamma
+ D_i \delta
+ G_i \eta
+ T_i \theta
+ \varepsilon_i.
$$

Donde:

- $X^{num}_i$: variables continuas estandarizadas.
- $B_i$: variables binarias 0/1.
- $D_i$: dummies de variables categóricas.
- $G_i$: fixed effects geográficos, si corresponde.
- $T_i$: fixed effects temporales, si corresponde.

El objetivo de esta notebook es hacer explícito qué términos existen en cada especificación y cómo se interpreta cada coeficiente.


In [4]:
def equation_statement(info: dict) -> str:
    exp = info["experiment"] if "experiment" in info else None
    include_blocks = as_list(info.get("include_blocks"))
    fixed_effects = as_list(info.get("fixed_effects"))
    fe_terms = []
    for spec in fixed_effects:
        if isinstance(spec, Mapping):
            name = str(spec.get("name") or "fe")
            cols = " × ".join(str(c) for c in as_list(spec.get("columns")))
            fe_terms.append(f"{name}[{cols}]")

    parts = [
        "log10(P47T_i) = α",
        "+ X_num,i β",
        "+ B_i γ",
        "+ D_i δ",
    ]
    if fe_terms:
        parts.append("+ FE_i θ")
    parts.append("+ ε_i")

    return " ".join(parts) + f" | bloques={include_blocks}; FE={fe_terms if fe_terms else 'none'}"


equation_statements = []
for exp, info in RUN_INFO.items():
    if not info["found"]:
        continue
    equation_statements.append({
        "experiment": exp,
        "label": info["label"],
        "equation_statement": equation_statement({"experiment": exp, **info}),
    })

equation_statements = pd.DataFrame(equation_statements)
equation_statements.to_csv(TABLE_DIR / "T3_equation_statements.csv", index=False)

for _, row in equation_statements.iterrows():
    print(f"- {row['label']}: {row['equation_statement']}")


- Core OLS limpio: log10(P47T_i) = α + X_num,i β + B_i γ + D_i δ + ε_i | bloques=['demographic', 'education', 'labor', 'housing_household']; FE=none
- Core + aglo + año + trimestre FE: log10(P47T_i) = α + X_num,i β + B_i γ + D_i δ + FE_i θ + ε_i | bloques=['demographic', 'education', 'labor', 'housing_household']; FE=['aglo_fe[AGLOMERADO]', 'year_fe[ANO4]', 'quarter_fe[TRIMESTRE]']
- Core + pirámide del hogar: log10(P47T_i) = α + X_num,i β + B_i γ + D_i δ + ε_i | bloques=['demographic', 'education', 'labor', 'housing_household', 'household_pyramid']; FE=none
- Core sin pirámide (legacy): log10(P47T_i) = α + X_num,i β + B_i γ + D_i δ + ε_i | bloques=['demographic', 'education', 'labor', 'housing_household']; FE=none
- Core + aglomerado FE (legacy/aux): log10(P47T_i) = α + X_num,i β + B_i γ + D_i δ + FE_i θ + ε_i | bloques=['demographic', 'education', 'labor', 'housing_household']; FE=['aglo_fe[AGLOMERADO]']
- Core + año FE (aux): log10(P47T_i) = α + X_num,i β + B_i γ + D_i δ + FE_i θ + 

## 04. De variables crudas a matriz de diseño

Esta sección es central. Un coeficiente OLS no siempre corresponde a “una variable cruda”. Puede corresponder a:

- una variable continua estandarizada;
- una dummy binaria;
- una categoría de una variable one-hot;
- un fixed effect.

Por eso usamos `design_matrix_features.csv` y `feature_metadata.json`.


In [5]:
def add_experiment_columns(df: pd.DataFrame, exp: str, info: dict) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out.insert(0, "experiment", exp)
    out.insert(1, "experiment_label", info["label"])
    out.insert(2, "run_id", info.get("run_id"))
    return out


def read_diag(exp: str, filename: str) -> pd.DataFrame:
    info = RUN_INFO[exp]
    if not info["found"]:
        return pd.DataFrame()
    return read_csv_if_exists(info["diagnostics_dir"] / filename)


design_parts = []
feature_metadata_parts = []

for exp, info in RUN_INFO.items():
    if not info["found"]:
        continue

    design = read_diag(exp, "design_matrix_features.csv")
    if not design.empty:
        design_parts.append(add_experiment_columns(design, exp, info))

    metadata = info.get("feature_metadata") or {}
    if isinstance(metadata, Mapping):
        rows = metadata.get("features") or metadata.get("feature_metadata") or []
        if isinstance(rows, list) and rows:
            feature_metadata_parts.append(
                add_experiment_columns(pd.DataFrame(rows), exp, info)
            )

design_matrix = pd.concat(design_parts, ignore_index=True) if design_parts else pd.DataFrame()
feature_metadata = pd.concat(feature_metadata_parts, ignore_index=True) if feature_metadata_parts else pd.DataFrame()

if design_matrix.empty and feature_metadata.empty:
    print("No design_matrix_features.csv or feature_metadata.json found. Re-run experiments with the upgraded backend.")
else:
    print("design_matrix:", design_matrix.shape)
    print("feature_metadata:", feature_metadata.shape)

# A compact audit table.
audit_source = design_matrix.copy()
if audit_source.empty and not feature_metadata.empty:
    audit_source = feature_metadata.rename(columns={"column": "raw_feature"}).copy()
    if "transformed_feature" not in audit_source.columns and "raw_feature" in audit_source.columns:
        audit_source["transformed_feature"] = audit_source["raw_feature"]

if not audit_source.empty:
    for col in ["feature_type", "is_fixed_effect"]:
        if col not in audit_source.columns:
            audit_source[col] = pd.NA

    design_audit = (
        audit_source
        .assign(
            is_fixed_effect=lambda d: d["is_fixed_effect"].fillna(False).astype(bool),
            feature_type=lambda d: d["feature_type"].fillna("unknown"),
        )
        .groupby(["experiment", "experiment_label"], dropna=False)
        .agg(
            n_raw_features=("raw_feature", "nunique"),
            n_transformed_features=("transformed_feature", "nunique"),
            n_continuous=("feature_type", lambda s: int((s == "continuous_numeric").sum())),
            n_binary=("feature_type", lambda s: int((s == "binary").sum())),
            n_categorical_or_dummy=("feature_type", lambda s: int(s.astype(str).str.contains("categorical", na=False).sum())),
            n_fixed_effect=("is_fixed_effect", lambda s: int(s.sum())),
        )
        .reset_index()
    )
else:
    design_audit = pd.DataFrame()

design_audit.to_csv(TABLE_DIR / "T4_design_matrix_audit.csv", index=False)
design_audit


design_matrix: (649, 15)
feature_metadata: (0, 0)


,experiment,experiment_label,n_raw_features,n_transformed_features,n_continuous,n_binary,n_categorical_or_dummy,n_fixed_effect
0,ols_core,Core OLS limpio,21,80,2,0,78,0
1,ols_core_aglo_fe,Core + aglomerado FE (legacy/aux),22,112,2,0,78,32
2,ols_core_aglo_plus_time_fe,Core + aglo + año + trimestre FE,24,118,2,0,78,38
3,ols_core_quarter_fe,Core + trimestre FE (aux),22,83,2,0,78,3
4,ols_core_with_pyramid,Core + pirámide del hogar,28,87,9,0,78,0
5,ols_core_year_fe,Core + año FE (aux),22,83,2,0,78,3
6,ols_core_year_plus_quarter_fe,Core + año + trimestre FE (aux),23,86,2,0,78,6


## 05. Diccionario de interpretación de columnas

Esta tabla traduce cada tipo de columna a su unidad interpretativa.

No hay que leer igual una variable continua estandarizada, una dummy binaria, una categoría one-hot y un FE.


In [6]:
feature_type_dictionary = pd.DataFrame([
    {
        "feature_type": "continuous_numeric",
        "preprocessing": "imputación por mediana + StandardScaler",
        "interpretation_unit": "+1 desviación estándar",
        "coefficient_reading": "diferencia condicional en log10 ingreso por +1 SD",
    },
    {
        "feature_type": "binary",
        "preprocessing": "imputación por moda / valor frecuente + 0/1 sin scaling",
        "interpretation_unit": "1 contra 0",
        "coefficient_reading": "diferencia condicional entre indicador activo e inactivo",
    },
    {
        "feature_type": "categorical",
        "preprocessing": "one-hot encoding con categoría omitida",
        "interpretation_unit": "categoría contra referencia",
        "coefficient_reading": "diferencia condicional respecto de la categoría omitida",
    },
    {
        "feature_type": "fixed_effect",
        "preprocessing": "one-hot encoding con grupo omitido",
        "interpretation_unit": "grupo contra grupo de referencia",
        "coefficient_reading": "diferencia residual de grupo respecto del grupo omitido",
    },
])

feature_type_dictionary.to_csv(TABLE_DIR / "T5_feature_type_interpretation_dictionary.csv", index=False)
feature_type_dictionary


,feature_type,preprocessing,interpretation_unit,coefficient_reading
0,continuous_numeric,imputación por mediana + StandardScaler,+1 desviación estándar,diferencia condicional en log10 ingreso por +1 SD
1,binary,imputación por moda / valor frecuente + 0/1 si...,1 contra 0,diferencia condicional entre indicador activo ...
2,categorical,one-hot encoding con categoría omitida,categoría contra referencia,diferencia condicional respecto de la categorí...
3,fixed_effect,one-hot encoding con grupo omitido,grupo contra grupo de referencia,diferencia residual de grupo respecto del grup...


## 06. Categorías omitidas y niveles de referencia

Todo coeficiente de una dummy se interpreta **contra una categoría omitida**. Sin esta tabla, la interpretación de coeficientes categóricos y fixed effects no es defendible.

La interpretación correcta no es:

> “Esta categoría aumenta el ingreso”.

Sino:

> “Esta categoría se asocia con un ingreso esperado mayor/menor que la categoría de referencia, manteniendo constantes las demás variables incluidas”.


In [7]:
reference_parts = []

for exp, info in RUN_INFO.items():
    if not info["found"]:
        continue

    onehot = read_diag(exp, "onehot_reference_levels.csv")
    fe_ref = read_diag(exp, "fixed_effect_reference_levels.csv")

    if not onehot.empty:
        onehot["reference_kind"] = "onehot"
        reference_parts.append(add_experiment_columns(onehot, exp, info))


# --------------------------------------------------------------------
# ValueError                                Traceback (most recent call last)
# /tmp/ipykernel_755272/2162248696.py in ?()
#       8     fe_ref = read_diag(exp, "fixed_effect_reference_levels.csv")
#       9 
#      10     if not onehot.empty:
#      11         onehot["reference_kind"] = "onehot"
# ---> 12         reference_parts.append(add_experiment_columns(onehot, exp, info))
#      13 
#      14     if not fe_ref.empty:
#      15         fe_ref["reference_kind"] = "fixed_effect"

# /tmp/ipykernel_755272/1966576065.py in ?(df, exp, info)
#       3         return df
#       4     out = df.copy()
#       5     out.insert(0, "experiment", exp)
#       6     out.insert(1, "experiment_label", info["label"])
# ----> 7     out.insert(2, "run_id", info.get("run_id"))
#       8     return out

# ~/anaconda3/envs/new_env/lib/python3.11/site-packages/pandas/core/frame.py in ?(self, loc, column, value, allow_duplicates)
#    5617                 "'self.flags.allows_duplicate_labels' is False."
#    5618             )
#    5619         if not allow_duplicates and column in self.columns:
#    5620             # Should this be a different kind of error??
# -> 5621             raise ValueError(f"cannot insert {column}, already exists")
#    5622         if not is_integer(loc):
#    5623             raise TypeError("loc must be int")
#    5624         # convert non stdlib ints to satisfy typing checks

# ValueError: cannot insert run_id, already exists




    if not fe_ref.empty:
        fe_ref["reference_kind"] = "fixed_effect"
        reference_parts.append(add_experiment_columns(fe_ref, exp, info))

reference_levels = pd.concat(reference_parts, ignore_index=True) if reference_parts else pd.DataFrame()

if not reference_levels.empty:
    preferred_ref_cols = [
        "experiment",
        "experiment_label",
        "model",
        "reference_kind",
        "raw_feature",
        "feature_family",
        "feature_type",
        "transformer",
        "reference_level",
        "is_fixed_effect",
        "fe_name",
        "source_columns",
    ]
    ordered = [c for c in preferred_ref_cols if c in reference_levels.columns]
    ordered += [c for c in reference_levels.columns if c not in ordered]
    reference_levels = reference_levels[ordered].drop_duplicates()
    reference_levels.to_csv(TABLE_DIR / "T6_reference_levels.csv", index=False)
else:
    print("No reference level artifacts found. Re-run experiments with upgraded coefficient diagnostics.")

reference_levels.head(80)


ValueError: cannot insert run_id, already exists

## 07. Conversión de coeficientes en `log10(P47T)`

El target está en logaritmo base 10. Por lo tanto, un coeficiente \(\beta\) tiene una interpretación multiplicativa:

\[
\text{factor} = 10^\beta
\]

\[
\text{cambio porcentual} = 100 \times (10^\beta - 1)
\]

Esta tabla fija es un ancla para leer los coeficientes sin exagerar magnitudes.


In [ ]:
conversion_table = pd.DataFrame({"coef_log10": [-0.20, -0.10, -0.05, -0.01, 0.01, 0.05, 0.10, 0.20]})
conversion_table["income_factor"] = 10 ** conversion_table["coef_log10"]
conversion_table["pct_change"] = 100 * (conversion_table["income_factor"] - 1)
conversion_table.to_csv(TABLE_DIR / "T7_log10_coefficient_conversion.csv", index=False)
conversion_table.round({"coef_log10": 3, "income_factor": 3, "pct_change": 1})


## 08. Carga y normalización de coeficientes enriquecidos

Esta sección usa `coefficients_transformed_all.csv`. Si no existe, la notebook puede leer algunos artifacts legacy, pero la interpretación sustantiva completa requiere el backend actualizado.


In [ ]:
def read_coefficients_for_experiment(exp: str) -> pd.DataFrame:
    info = RUN_INFO[exp]
    if not info["found"]:
        return pd.DataFrame()

    rich = read_diag(exp, "coefficients_transformed_all.csv")
    if not rich.empty:
        return add_experiment_columns(rich, exp, info)

    # Legacy fallback: useful only for rough audit.
    candidates = []
    diag = info["diagnostics_dir"]
    if diag.exists():
        for path in sorted(diag.glob("*coefficients*.csv")):
            if path.name in {
                "onehot_reference_levels.csv",
                "fixed_effect_reference_levels.csv",
                "coefficient_family_summary.csv",
                "top_coefficients_by_family.csv",
            }:
                continue
            df = pd.read_csv(path)
            df["source_file"] = path.name
            candidates.append(df)

    if not candidates:
        return pd.DataFrame()

    legacy = pd.concat(candidates, ignore_index=True)
    feature_col = next((c for c in ["transformed_feature", "feature", "feature_name", "term"] if c in legacy.columns), None)
    coef_col = next((c for c in ["coef_log10", "coefficient", "coef", "estimate"] if c in legacy.columns), None)
    if feature_col is None or coef_col is None:
        return pd.DataFrame()

    out = legacy.rename(columns={feature_col: "transformed_feature", coef_col: "coef_log10"}).copy()
    out["coef_log10"] = pd.to_numeric(out["coef_log10"], errors="coerce")
    out["abs_coef_log10"] = out["coef_log10"].abs()
    out["income_factor"] = 10 ** out["coef_log10"]
    out["pct_change"] = 100 * (out["income_factor"] - 1)
    if "raw_feature" not in out.columns:
        out["raw_feature"] = out["transformed_feature"].astype(str).str.replace(r"^[a-zA-Z0-9_]+__", "", regex=True)
    if "feature_family" not in out.columns:
        out["feature_family"] = "unknown_legacy"
    if "feature_type" not in out.columns:
        out["feature_type"] = "unknown_legacy"
    if "is_fixed_effect" not in out.columns:
        out["is_fixed_effect"] = out["transformed_feature"].astype(str).str.contains("fe_", regex=False)
    return add_experiment_columns(out, exp, info)


coef_parts = []
for exp in EXPERIMENT_ORDER:
    df = read_coefficients_for_experiment(exp)
    if not df.empty:
        coef_parts.append(df)

coefficients = pd.concat(coef_parts, ignore_index=True) if coef_parts else pd.DataFrame()

if coefficients.empty:
    raise RuntimeError("No coefficient artifacts found. Re-run OLS experiments with coefficient diagnostics enabled.")

# Normalize expected fields.
for col, default in [
    ("model", "LinearRegression"),
    ("feature_family", "unknown"),
    ("feature_type", "unknown"),
    ("raw_feature", pd.NA),
    ("transformed_feature", pd.NA),
    ("level", pd.NA),
    ("reference_level", pd.NA),
    ("is_fixed_effect", False),
    ("fe_name", pd.NA),
]:
    if col not in coefficients.columns:
        coefficients[col] = default

coefficients["coef_log10"] = pd.to_numeric(coefficients["coef_log10"], errors="coerce")
coefficients = coefficients.dropna(subset=["coef_log10"]).copy()
coefficients["abs_coef_log10"] = coefficients.get("abs_coef_log10", coefficients["coef_log10"].abs())
coefficients["abs_coef_log10"] = pd.to_numeric(coefficients["abs_coef_log10"], errors="coerce")
coefficients["income_factor"] = 10 ** coefficients["coef_log10"]
coefficients["pct_change"] = 100 * (coefficients["income_factor"] - 1)
coefficients["is_fixed_effect"] = coefficients["is_fixed_effect"].fillna(False).astype(bool)

# Prefer LinearRegression for this notebook when several coefficient-bearing models exist.
ols_coefficients = coefficients[coefficients["model"].astype(str).eq("LinearRegression")].copy()
if ols_coefficients.empty:
    ols_coefficients = coefficients.copy()

coefficients.to_csv(TABLE_DIR / "T8_coefficients_transformed_all_loaded.csv", index=False)

print("coefficients:", coefficients.shape)
print("ols_coefficients:", ols_coefficients.shape)
ols_coefficients.head(20)


## 09. Resumen por familia de coeficientes

Esta tabla sirve como auditoría de escala, no como conclusión sustantiva. Un `sum_abs_coef` alto no implica “familia causalmente importante”. Indica que la familia aporta muchas columnas o coeficientes grandes en la matriz estimada.


In [ ]:
family_summary_parts = []
for exp, info in RUN_INFO.items():
    if not info["found"]:
        continue
    fs = read_diag(exp, "coefficient_family_summary.csv")
    if not fs.empty:
        family_summary_parts.append(add_experiment_columns(fs, exp, info))

if family_summary_parts:
    coefficient_family_summary = pd.concat(family_summary_parts, ignore_index=True)
else:
    coefficient_family_summary = (
        ols_coefficients
        .groupby(["experiment", "experiment_label", "model", "feature_family", "is_fixed_effect"], dropna=False)
        .agg(
            n_coefficients=("coef_log10", "count"),
            mean_abs_coef_log10=("abs_coef_log10", "mean"),
            median_abs_coef_log10=("abs_coef_log10", "median"),
            p90_abs_coef_log10=("abs_coef_log10", lambda s: float(s.quantile(0.90))),
            max_abs_coef_log10=("abs_coef_log10", "max"),
            sum_abs_coef_log10=("abs_coef_log10", "sum"),
        )
        .reset_index()
    )

coefficient_family_summary = coefficient_family_summary.sort_values(
    [c for c in ["experiment", "is_fixed_effect", "sum_abs_coef_log10"] if c in coefficient_family_summary.columns],
    ascending=[True, True, False][:len([c for c in ["experiment", "is_fixed_effect", "sum_abs_coef_log10"] if c in coefficient_family_summary.columns])]
)

coefficient_family_summary.to_csv(TABLE_DIR / "T9_coefficient_family_summary.csv", index=False)
coefficient_family_summary.round(5).head(80)


## 10. Coeficientes no-FE seleccionados

No mostramos “top 50 coeficientes” sin contexto. Eso mezcla dummies raras, variables redundantes y categorías con distinto soporte.

La regla de esta sección:

- excluir fixed effects;
- separar por familia;
- reportar \(\beta\), factor multiplicativo y cambio porcentual;
- preservar referencia cuando exista;
- recordar que son asociaciones condicionales, no efectos causales.


In [ ]:
def clean_level(value):
    if pd.isna(value):
        return ""
    return str(value)


def coefficient_key(row: pd.Series) -> str:
    raw = str(row.get("raw_feature", ""))
    level = clean_level(row.get("level"))
    ftype = str(row.get("feature_type", ""))
    if level:
        return f"{raw}::{level}"
    return f"{raw}::{ftype}"


def coefficient_interpretation(row: pd.Series) -> str:
    coef = row.get("coef_log10")
    pct = row.get("pct_change")
    raw = row.get("raw_feature")
    level = row.get("level")
    ref = row.get("reference_level")
    ftype = str(row.get("feature_type", ""))
    if pd.isna(coef):
        return ""

    pct_text = f"{pct:+.1f}%" if pd.notna(pct) else "NA"
    if bool(row.get("is_fixed_effect", False)):
        return f"{level} vs referencia {ref}: asociación residual {pct_text} en ingreso esperado."

    if ftype == "continuous_numeric":
        return f"{raw}: +1 SD se asocia con {pct_text} en ingreso esperado, condicional en controles."
    if ftype == "binary":
        return f"{raw}: 1 vs 0 se asocia con {pct_text} en ingreso esperado, condicional en controles."
    if pd.notna(level) and str(level):
        return f"{raw}={level} vs referencia {ref}: asociación condicional {pct_text}."
    return f"{raw}: asociación condicional {pct_text}."


non_fe = ols_coefficients[~ols_coefficients["is_fixed_effect"]].copy()
non_fe["coef_key"] = non_fe.apply(coefficient_key, axis=1)
non_fe["interpretation"] = non_fe.apply(coefficient_interpretation, axis=1)

selected_non_fe = (
    non_fe
    .sort_values("abs_coef_log10", ascending=False)
    .groupby(["experiment", "experiment_label", "feature_family"], dropna=False)
    .head(8)
    .reset_index(drop=True)
)

preferred_cols = [
    "experiment",
    "experiment_label",
    "model",
    "feature_family",
    "feature_type",
    "raw_feature",
    "level",
    "reference_level",
    "coef_log10",
    "income_factor",
    "pct_change",
    "interpretation",
]
selected_non_fe = selected_non_fe[[c for c in preferred_cols if c in selected_non_fe.columns]]
selected_non_fe.to_csv(TABLE_DIR / "T10_selected_non_fe_coefficients_by_family.csv", index=False)

selected_non_fe.round({"coef_log10": 4, "income_factor": 3, "pct_change": 1}).head(120)


## 11. Estabilidad: core limpio vs core ajustado con FE

Esta es probablemente la comparación más útil de la notebook.

Pregunta:

> ¿Las asociaciones individuales principales sobreviven cuando agregamos heterogeneidad aditiva por aglomerado, año y trimestre?

Si un coeficiente cambia mucho al agregar FE, no está “mal”, pero sí debemos interpretarlo como sensible a estructura territorial/temporal.


In [ ]:
def stability_table(core_exp="ols_core", adjusted_exp="ols_core_aglo_plus_time_fe") -> pd.DataFrame:
    core = non_fe[non_fe["experiment"].eq(core_exp)].copy()
    adj = non_fe[non_fe["experiment"].eq(adjusted_exp)].copy()

    if core.empty or adj.empty:
        return pd.DataFrame()

    core["coef_key"] = core.apply(coefficient_key, axis=1)
    adj["coef_key"] = adj.apply(coefficient_key, axis=1)

    keep = [
        "coef_key",
        "model",
        "feature_family",
        "feature_type",
        "raw_feature",
        "level",
        "reference_level",
        "coef_log10",
        "pct_change",
        "abs_coef_log10",
    ]
    core = core[[c for c in keep if c in core.columns]].rename(columns={
        "coef_log10": "coef_core",
        "pct_change": "pct_core",
        "abs_coef_log10": "abs_coef_core",
        "reference_level": "reference_level_core",
    })
    adj = adj[[c for c in keep if c in adj.columns]].rename(columns={
        "coef_log10": "coef_adjusted",
        "pct_change": "pct_adjusted",
        "abs_coef_log10": "abs_coef_adjusted",
        "reference_level": "reference_level_adjusted",
    })

    merged = core.merge(
        adj,
        on=["coef_key", "model", "feature_family", "feature_type", "raw_feature", "level"],
        how="inner",
        suffixes=("", "_adjusted_meta"),
    )
    if merged.empty:
        return merged

    merged["delta_coef_adjusted_minus_core"] = merged["coef_adjusted"] - merged["coef_core"]
    merged["abs_delta_coef"] = merged["delta_coef_adjusted_minus_core"].abs()
    merged["delta_pct_points"] = merged["pct_adjusted"] - merged["pct_core"]
    return merged.sort_values("abs_delta_coef", ascending=False).reset_index(drop=True)


coef_stability = stability_table()
coef_stability.to_csv(TABLE_DIR / "T11_coefficient_stability_core_vs_adjusted.csv", index=False)

if coef_stability.empty:
    print("Coefficient stability table unavailable. Need both ols_core and ols_core_aglo_plus_time_fe with enriched coefficient artifacts.")
else:
    display_cols = [
        "feature_family",
        "feature_type",
        "raw_feature",
        "level",
        "reference_level_core",
        "reference_level_adjusted",
        "coef_core",
        "pct_core",
        "coef_adjusted",
        "pct_adjusted",
        "delta_coef_adjusted_minus_core",
        "delta_pct_points",
    ]
    coef_stability[[c for c in display_cols if c in coef_stability.columns]].round({
        "coef_core": 4,
        "pct_core": 1,
        "coef_adjusted": 4,
        "pct_adjusted": 1,
        "delta_coef_adjusted_minus_core": 4,
        "delta_pct_points": 1,
    }).head(80)


In [ ]:
if MAKE_OPTIONAL_FIGURES and not coef_stability.empty:
    import matplotlib.pyplot as plt

    plot_df = coef_stability.copy()
    # Avoid unreadable plots with huge numbers of tiny dummies.
    plot_df = plot_df.sort_values("abs_coef_core", ascending=False).head(80)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(plot_df["coef_core"], plot_df["coef_adjusted"], alpha=0.75)
    lim = max(plot_df["coef_core"].abs().max(), plot_df["coef_adjusted"].abs().max())
    ax.plot([-lim, lim], [-lim, lim], linestyle="--", linewidth=1)
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)
    ax.set_xlabel("Coeficiente log10 — core limpio")
    ax.set_ylabel("Coeficiente log10 — core + FE")
    ax.set_title("Estabilidad de coeficientes no-FE")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "F1_coefficient_stability_core_vs_adjusted.png", dpi=160)
    plt.show()
else:
    print("Optional figure disabled. Tables are the default output for this notebook.")


## 12. Fixed effects: interpretación separada

Los FE no son “variables individuales”. Representan diferencias residuales de grupo respecto de una categoría omitida, después de controlar por las covariables incluidas.

Por eso no se mezclan con los coeficientes no-FE.

- FE geográfico: diferencia residual del aglomerado contra aglomerado de referencia.
- FE temporal: diferencia residual del año/trimestre contra período de referencia.
- No son efectos causales de vivir en una ciudad ni efectos macro puros.


In [ ]:
def fe_level_label(row: pd.Series) -> str:
    level = clean_level(row.get("level"))
    fe_name = str(row.get("fe_name", "")).lower()
    raw = str(row.get("raw_feature", "")).lower()
    source_columns = safe_json_load(row.get("source_columns")) if "source_columns" in row.index else None
    source_text = " ".join(str(x).lower() for x in as_list(source_columns))

    if "aglo" in fe_name or "aglo" in raw or "aglomerado" in source_text:
        return AGLO_LABELS.get(level, level)
    if "year" in fe_name or "ano4" in source_text:
        return f"Año {level}"
    if "quarter" in fe_name or "trimestre" in source_text:
        return f"Trimestre {level}"
    return level


fe_coefs = ols_coefficients[ols_coefficients["is_fixed_effect"]].copy()
if fe_coefs.empty:
    print("No fixed-effect coefficients found in selected OLS coefficient artifacts.")
    fixed_effect_selected = pd.DataFrame()
else:
    fe_coefs["level_label"] = fe_coefs.apply(fe_level_label, axis=1)
    fe_coefs["interpretation"] = fe_coefs.apply(coefficient_interpretation, axis=1)

    fixed_effect_selected = (
        fe_coefs
        .sort_values("abs_coef_log10", ascending=False)
        .groupby(["experiment", "experiment_label", "fe_name"], dropna=False)
        .head(12)
        .reset_index(drop=True)
    )

    fixed_effect_cols = [
        "experiment",
        "experiment_label",
        "model",
        "fe_name",
        "raw_feature",
        "source_columns",
        "level",
        "level_label",
        "reference_level",
        "coef_log10",
        "income_factor",
        "pct_change",
        "interpretation",
    ]
    fixed_effect_selected = fixed_effect_selected[[c for c in fixed_effect_cols if c in fixed_effect_selected.columns]]
    fixed_effect_selected.to_csv(TABLE_DIR / "T12_selected_fixed_effect_coefficients.csv", index=False)

fixed_effect_selected.round({"coef_log10": 4, "income_factor": 3, "pct_change": 1}).head(120)


## 13. Sensibilidad con pirámide del hogar

La pirámide del hogar no forma parte del benchmark limpio. Puede ser predictivamente útil, pero su interpretación es composicional y puede introducir redundancias con edad, composición del hogar y ciclo de vida.

La pregunta correcta no es “¿pyramid mejora algo?”, sino:

> ¿Agregar composición del hogar cambia de forma relevante los coeficientes sustantivos del core?


In [ ]:
coef_stability_pyramid = pd.DataFrame()

if RUN_INFO.get("ols_core_with_pyramid", {}).get("found"):
    pyramid_exp = "ols_core_with_pyramid"
elif RUN_INFO.get("ols_core_no_pyramid", {}).get("found") and RUN_INFO.get("ols_core", {}).get("found"):
    # Legacy fallback: in old runs ols_core may have included pyramid.
    pyramid_exp = "ols_core"
else:
    pyramid_exp = None

if pyramid_exp and RUN_INFO["ols_core"]["found"] and pyramid_exp != "ols_core":
    coef_stability_pyramid = stability_table("ols_core", pyramid_exp)
elif pyramid_exp == "ols_core":
    print("Legacy pyramid comparison detected but ambiguous: current ols_core is also the clean benchmark in the new suite.")
else:
    print("No pyramid sensitivity run found.")

if not coef_stability_pyramid.empty:
    coef_stability_pyramid.to_csv(TABLE_DIR / "T13_coefficient_stability_core_vs_pyramid.csv", index=False)
    display(coef_stability_pyramid[[
        c for c in [
            "feature_family",
            "feature_type",
            "raw_feature",
            "level",
            "coef_core",
            "pct_core",
            "coef_adjusted",
            "pct_adjusted",
            "delta_coef_adjusted_minus_core",
            "delta_pct_points",
        ]
        if c in coef_stability_pyramid.columns
    ]].round(4).head(80))


## 14. Red flags de interpretación

Esta sección produce notas automáticas, pero el punto sustantivo es fijo:

1. **No causalidad**: los coeficientes son asociaciones condicionales predictivas.
2. **Dummies con referencia**: todo coeficiente categórico depende de la categoría omitida.
3. **Educación puede ser redundante** si se incluyen simultáneamente nivel, completitud y variables agregadas.
4. **Trabajo es endógeno al proceso de ingreso**: muy predictivo, pero no exógeno.
5. **Edad lineal no es perfil de ciclo de vida**.
6. **Fixed effects no son efectos causales de lugar o tiempo**.
7. **Pyramid es sensibilidad**, no default.


In [ ]:
notes = []

# Run-level notes.
for _, row in spec_policy.iterrows():
    notes.append(
        f"{row['experiment']}: role={row['role']}; "
        f"uses_household_pyramid={row['uses_household_pyramid']}; FE={row['fixed_effects']}."
    )

# Artifact completeness.
for artifact_name, frame in [
    ("design_matrix_features", design_matrix),
    ("feature_metadata", feature_metadata),
    ("reference_levels", reference_levels),
    ("coefficients_transformed_all", coefficients),
    ("coefficient_stability_core_vs_adjusted", coef_stability),
    ("fixed_effect_selected", fixed_effect_selected),
]:
    notes.append(f"{artifact_name}: {'available' if not frame.empty else 'missing/empty'}.")

# Education redundancy check.
education_names = set()
if not non_fe.empty:
    education_rows = non_fe[non_fe["feature_family"].astype(str).str.contains("educ", case=False, na=False)]
    education_names = set(education_rows["raw_feature"].dropna().astype(str).unique())

education_warning_terms = {"P09", "P10", "P0910", "Max_Nivel_Educativo", "NIVEL_ED"}
present_warning_terms = sorted(t for t in education_warning_terms if t in education_names)
if len(present_warning_terms) >= 2:
    notes.append(
        "Education interpretation warning: multiple education codings appear together "
        f"({present_warning_terms}). Coefficients should not be read as clean returns to schooling."
    )

if not reference_levels.empty:
    notes.append("Reference levels are available; categorical and FE coefficients can be interpreted against explicit omitted categories.")
else:
    notes.append("Reference levels are missing; categorical and FE coefficients should not be interpreted substantively until references are exported.")

notes.append(
    "General interpretation rule: coefficient beta in log10 target corresponds to pct_change = 100 * (10**beta - 1)."
)
notes.append(
    "Causal caveat: coefficients describe conditional associations within the fitted OLS design, not causal effects."
)

diagnostic_notes = "\n".join(f"- {note}" for note in notes)
(TABLE_DIR / "T14_ols_equation_interpretation_notes.txt").write_text(diagnostic_notes, encoding="utf-8")
print(diagnostic_notes)


## 15. Cierre interpretativo

La contribución de esta notebook no es encontrar “el mejor coeficiente”. Es hacer trazable la especificación OLS:

- qué variables entraron;
- cómo se transformaron;
- qué categoría quedó omitida;
- en qué unidad se interpreta cada coeficiente;
- qué cambia al agregar FE;
- qué no debe leerse causalmente.

La frase guía para leer cualquier número de esta notebook es:

> **Este coeficiente mide una asociación condicional en log10 ingreso, respecto de una unidad o categoría de referencia específica, manteniendo constantes las demás variables incluidas en la especificación.**


In [ ]:
outputs = sorted(TABLE_DIR.glob("*.csv")) + sorted(TABLE_DIR.glob("*.txt"))
print("Outputs written:")
for path in outputs:
    print("-", path.relative_to(OUTPUT_DIR))

summary = {
    "output_dir": str(OUTPUT_DIR),
    "tables": [str(path.relative_to(OUTPUT_DIR)) for path in outputs],
    "figures": [str(path.relative_to(OUTPUT_DIR)) for path in sorted(FIG_DIR.glob("*"))],
}
(OUTPUT_DIR / "notebook_15_outputs_manifest.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
summary
